In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append("../../modules")
from PPBAffinity.process_data import *

# Original processing

In [2]:
affinity_data = pd.read_excel('../../data/ppb_affinity/PPB-Affinity.xlsx',
                              usecols=['PDB', 'Source Data Set', 'Mutations', 'Ligand Chains', 'Receptor Chains',
                                        'KD(M)', 'Subgroup', 'Resolution(Å)', 'Complex ID', 'PDB Release Date'],
                              dtype={"PDB": str})
affinity_data.rename(columns={'Source Data Set': 'source', 'KD(M)': 'KD',
                              'Mutations': 'mutstr', 'PDB': 'pdb',
                              'Ligand Chains': 'ligand', 'Receptor Chains': 'receptor', 'Resolution(Å)': 'resolution',
                              'Complex ID': 'complex_id', 'PDB Release Date': 'release_date'}, inplace=True)

affinity_data['dG'] = (8.314/4184)*(273.15 + 25.0) * np.log(affinity_data['KD'])
affinity_data = affinity_data[~(affinity_data.ligand.str.len() + affinity_data.receptor.str.len() > 26)]
affinity_data.reset_index(drop=True, inplace=True)
# affinity_data['mutstr'] = affinity_data['mutstr'].apply(mutstr_transform)

# Custom processing

In [3]:
affinity_data = affinity_data[affinity_data['resolution'] < 3.5]

In [4]:
from pathlib import Path

project_root = Path.cwd().parent.parent  # adjust as needed
data_path_base = project_root / "data"
affinity_data["pdb"] = affinity_data["pdb"].str.lower()


def get_pdb_path(row):
    if row["source"] == "SAbDab":
            pdb_file_name = f"{row['pdb']}.pdb"
    elif row["source"] == "PDBbind v2020":
        pdb_file_name = f"{row['pdb']}.ent.pdb"
    else:
        pdb_file_name = f"{row['pdb'].upper()}.pdb"
        
    pdb_path = data_path_base / "ppb_affinity" / "pdb" / row["source"] / pdb_file_name
    
    return pdb_path

affinity_data["pdb_path"] = affinity_data.apply(get_pdb_path, axis=1)

In [6]:
affinity_data

,source,complex_id,pdb,mutstr,ligand,receptor,KD,resolution,release_date,Subgroup,dG,pdb_path,hard_split
0,SKEMPI v2.0,"1A22:A, B::PMID=7504735",1a22,NaN,A,B,9.000000e-10,2.60,1998-04-29,NaN,-12.339961,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
1,SKEMPI v2.0,"1A4Y:A, B::PMID=9050852",1a4y,NaN,A,B,5.000000e-16,2.00,1998-10-14,NaN,-20.873223,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
2,SKEMPI v2.0,"1ACB:E, I::PMID=9048543",1acb,NaN,E,I,1.490000e-12,2.00,1993-10-31,NaN,-16.133798,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
3,SKEMPI v2.0,"1AHW:A, B, C::PMID=9480775",1ahw,NaN,"A, B",C,3.400000e-09,3.00,1998-02-25,NaN,-11.552512,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
4,SKEMPI v2.0,"1AK4:A, D::PMID=9223641",1ak4,NaN,A,D,1.200000e-05,2.36,1997-10-15,NaN,-6.712839,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12050,ATLAS,"4L3E:A, C, D, E:A_E166A:PMID=23736024",4l3e,A_E166A,"A, C","D, E",3.220000e-05,2.56,2014-06-11,TCR-pMHC,-6.128053,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
12051,ATLAS,"4L3E:A, C, D, E:A_Q155A, D_Y50A:PMID=23736024",4l3e,"A_Q155A, D_Y50A","A, C","D, E",7.710000e-05,2.56,2014-06-11,TCR-pMHC,-5.610762,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
12052,ATLAS,"4L3E:A, C, D, E:A_E166A, D_N52A:PMID=23736024",4l3e,"A_E166A, D_N52A","A, C","D, E",2.610000e-05,2.56,2014-06-11,TCR-pMHC,-6.252487,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train
12053,ATLAS,"4L3E:A, C, D, E:D_D26Y, E_L98W:PMID=22611242",4l3e,"D_D26Y, E_L98W","A, C","D, E",4.300000e-08,2.56,2014-06-11,TCR-pMHC,-10.049209,/home/ghandill/masterthesis_lilit_ghandilyan/d...,train


# Hard split

In [5]:
# Get unique PDBs
unique_pdbs = affinity_data['pdb'].unique()

# Shuffle the PDBs to randomize the hard_split
np.random.seed(42)  # for reproducibility
shuffled_pdbs = np.random.permutation(unique_pdbs)

# Calculate split indices for PDBs
n_pdbs = len(shuffled_pdbs)
train_end = int(n_pdbs * 0.95)
val_end = int(n_pdbs * 0.975)

# Assign each PDB to a split
pdb_to_split = {}
for i, pdb in enumerate(shuffled_pdbs):
    if i < train_end:
        pdb_to_split[pdb] = 'train'
    elif i < val_end:
        pdb_to_split[pdb] = 'val'
    else:
        pdb_to_split[pdb] = 'test'

# Map the split back to the dataframe
affinity_data['hard_split'] = affinity_data['pdb'].map(pdb_to_split)

# Verify no leakage
print("Split distribution:")
print(affinity_data['hard_split'].value_counts())
print(f"\nNumber of unique PDBs per hard_split:")
print(affinity_data.groupby('hard_split')['pdb'].nunique())
print(f"\nTotal datapoints per hard_split:")
print(affinity_data['hard_split'].value_counts())

# Verify no PDB appears in multiple hard_splits (should be empty)
pdb_hard_split_check = affinity_data.groupby('pdb')['hard_split'].nunique()
leaked_pdbs = pdb_hard_split_check[pdb_hard_split_check > 1]
print(f"\nPDBs in multiple hard_splits (should be 0): {len(leaked_pdbs)}")

Split distribution:
hard_split
train    10866
val        199
test       186
Name: count, dtype: int64

Number of unique PDBs per hard_split:
hard_split
test       68
train    2571
val        68
Name: pdb, dtype: int64

Total datapoints per hard_split:
hard_split
train    10866
val        199
test       186
Name: count, dtype: int64

PDBs in multiple hard_splits (should be 0): 0


In [ ]:
# import pandas as pd
# import networkx as nx

# list_df = pd.read_csv("../../data/ppb_affinity_old/list.csv")

# # df has columns: protein_id, chain_id, cluster
# G = nx.Graph()
# clusters = list_df['CLUSTER'].unique()
# G.add_nodes_from(clusters)

# # Connect clusters that share a protein
# for protein, grp in list_df.groupby('PDB_ID'):
#     protein_clusters = grp['CLUSTER'].unique()
#     for i in range(len(protein_clusters)):
#         for j in range(i+1, len(protein_clusters)):
#             G.add_edge(protein_clusters[i], protein_clusters[j])
            
# components = list(nx.connected_components(G))

# # Compute size (number of proteins in each component)
# component_sizes = []
# for comp in components:
#     proteins_in_comp = list_df[list_df['CLUSTER'].isin(comp)]['PDB_ID'].unique()
#     component_sizes.append({
#         'clusters': comp,
#         'n_proteins': len(proteins_in_comp)
#     })

# # Convert to dataframe for easy inspection
# comp_df = pd.DataFrame(component_sizes)
# print(comp_df)

# # Sort components by size (largest first)
# comp_df = comp_df.sort_values(by="n_proteins", ascending=False).reset_index(drop=True)

# train_proteins, val_proteins, test_proteins = set(), set(), set()
# train_cut = 0.91 * len(list_df['PDB_ID'].unique())
# val_cut   = 0.96 * len(list_df['PDB_ID'].unique())  # 80–90% = val, last 10% = test
# count = 0

# for _, row in comp_df.iterrows():
#     proteins = set(list_df[list_df['CLUSTER'].isin(row['clusters'])]['PDB_ID'].unique())
    
#     if count + len(proteins) <= train_cut:
#         train_proteins.update(proteins)
#     elif count + len(proteins) <= val_cut:
#         val_proteins.update(proteins)
#     else:
#         test_proteins.update(proteins)
    
#     count += len(proteins)

# # Assign split per protein
# def assign_split(protein):
#     if protein in train_proteins:
#         return 'train'
#     elif protein in val_proteins:
#         return 'val'
#     else:
#         return 'test'

# # affinity_data['split_hard'] = affinity_data['pdb'].apply(assign_split)
# # affinity_data["split_hard"].value_counts(normalize=True)

# Medium split

In [ ]:
affinity_data

In [ ]:
# Convert release date to datetime (if not already done)
affinity_data['release_date'] = pd.to_datetime(affinity_data['release_date'], errors='coerce')

# Sort by release date (oldest first)
affinity_data = affinity_data.sort_values('release_date').reset_index(drop=True)

# Compute split indices
n = len(affinity_data)
train_end = int(n * 0.95)
val_end = int(n * 0.975)   # 95% + 2.5%

# Create the split column
affinity_data['medium_split'] = np.where(
    affinity_data.index < train_end, 'train',
    np.where(affinity_data.index < val_end, 'val', 'test')
)


In [ ]:
affinity_data['medium_split'].value_counts(normalize=True)

# Easy split

In [ ]:
# Random split with same proportions
affinity_data['easy_split'] = (
    np.random.choice(
        ['train', 'val', 'test'],
        size=len(affinity_data),
        p=[0.95, 0.025, 0.025]
    )
)

In [ ]:
affinity_data

In [ ]:
# affinity_data.to_csv(data_path_base / "processed_data.csv")

In [ ]:
affinity_data['medium_split'].value_counts()

In [ ]:
affinity_data

In [ ]:
df = pd.read_csv(data_path_base / "processed_data.csv")

In [ ]:
affinity_data.to_csv(data_path_base / "processed_data.csv")

In [ ]:
df.release_date.max()

In [ ]:
df[df['medium_split'] == 'test']['release_date'].min()

In [ ]:
df[df['medium_split'] == 'test']['release_date'].max()

In [ ]:
affinity_data